In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("ventas").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.9


In [2]:
ruta = Path("../data/ventas_pentagrama_sim.csv").resolve().as_uri()
df = spark.read.csv(ruta, header=True, inferSchema=True)

df.show(5)
df.printSchema()
print(df.count(), len(df.columns))

+--------+--------------------+-------------------+------+------------+------+----------+
|id_venta|             cliente|           producto| color|    vendedor| monto|     fecha|
+--------+--------------------+-------------------+------+------------+------+----------+
|       1|          Pablo Melo|Persiana enrollable|Blanco| Sofia Gomez|442954|2026-05-01|
|       2|      Camilo Sánchez|Persiana enrollable| Negro|  Ana Torres|165815|2026-04-20|
|       3| Nataly Arias Tamayo|Persiana enrollable| Negro| Sofia Gomez|238817|2026-08-01|
|       4|   Jairo Ramos Rojas|Persiana enrollable|  Gris|  Ana Torres|173843|2026-07-19|
|       5|Johanna Martínez ...| Persiana veneciana|  Azul|Luis Ramirez|291424|2026-04-25|
+--------+--------------------+-------------------+------+------------+------+----------+
only showing top 5 rows

root
 |-- id_venta: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- color: string (nullable = true)
 |-

In [7]:
df[["cliente", "monto"]].show(6)


+--------------------+------+
|             cliente| monto|
+--------------------+------+
|          Pablo Melo|442954|
|      Camilo Sánchez|165815|
| Nataly Arias Tamayo|238817|
|   Jairo Ramos Rojas|173843|
|Johanna Martínez ...|291424|
|    Brayan Hernández|292855|
+--------------------+------+
only showing top 6 rows



In [8]:
df.select("cliente", "monto").show(6)

+--------------------+------+
|             cliente| monto|
+--------------------+------+
|          Pablo Melo|442954|
|      Camilo Sánchez|165815|
| Nataly Arias Tamayo|238817|
|   Jairo Ramos Rojas|173843|
|Johanna Martínez ...|291424|
|    Brayan Hernández|292855|
+--------------------+------+
only showing top 6 rows



In [9]:
df.filter((F.col("monto") > 300000) & (F.col("vendedor") == "Ana Torres")).show(5)

+--------+--------------------+-------------------+-----+----------+------+----------+
|id_venta|             cliente|           producto|color|  vendedor| monto|     fecha|
+--------+--------------------+-------------------+-----+----------+------+----------+
|      16|Carmen Teresa Ard...|   Cortina blackout| Azul|Ana Torres|409598|2026-08-24|
|      19|   Vilma Lizeth Caro|Persiana enrollable|Negro|Ana Torres|345923|2026-04-27|
|      22|Dolly María Perdo...|      Panel japones| Azul|Ana Torres|335931|2026-04-21|
|      24|Carlos Emiro Giraldo|   Cortina blackout|Negro|Ana Torres|310827|2026-09-18|
|      33|Esperanza Stella ...|Persiana enrollable|Negro|Ana Torres|430939|2026-08-21|
+--------+--------------------+-------------------+-----+----------+------+----------+
only showing top 5 rows



In [10]:
df.withColumn("monto_con_iva", F.col("monto") * 1.19).select("id_venta", "monto", "monto_con_iva").show(5)

+--------+------+------------------+
|id_venta| monto|     monto_con_iva|
+--------+------+------------------+
|       1|442954|         527115.26|
|       2|165815|197319.84999999998|
|       3|238817|         284192.23|
|       4|173843|206873.16999999998|
|       5|291424|         346794.56|
+--------+------+------------------+
only showing top 5 rows



In [11]:
(
    df.groupBy("producto")
    .agg(F.sum("monto").alias("total"), F.count("*").alias("ventas"), F.avg("monto").alias("promedio"))
    .orderBy(F.col("total").desc())
    .show()
)

+-------------------+--------+------+------------------+
|           producto|   total|ventas|          promedio|
+-------------------+--------+------+------------------+
|   Cortina blackout|15217931|    48| 317040.2291666667|
| Persiana veneciana|13134617|    43|305456.20930232556|
|Persiana enrollable|13047199|    43|303423.23255813954|
|      Panel japones|11590054|    36|321945.94444444444|
|Cortina translucida|10390813|    35| 296880.3714285714|
+-------------------+--------+------+------------------+



In [12]:
df.filter(F.col("cliente").isNull()).count()

8

In [13]:
filtrado = df.filter(F.col("monto") > 300000)
print("Filtro definido")
print(filtrado.count())

Filtro definido
113


In [14]:
import pandas as pd

pdf = pd.read_csv("../data/ventas_pentagrama_sim.csv")
print(pdf.groupby("producto")["monto"].sum().sort_values(ascending=False))

producto
Cortina blackout       15217931
Persiana veneciana     13134617
Persiana enrollable    13047199
Panel japones          11590054
Cortina translucida    10390813
Name: monto, dtype: int64


In [15]:
spark.stop()